# Программирование: Задание 9
## Кластеризация и Классификация в применении к данным стоматологической клиники

**ВКР:** «Информационная система стоматологической клиники»  
**Инструменты:** Python 3.11 · scikit-learn · matplotlib · seaborn · pandas  
**Платформа:** JupyterLab / Google Colab

---

### Структура ноутбука

| Часть | Раздел | Методы |
|---|---|---|
| **4** | Кластеризация (аналог Orange) | K-Means, DBSCAN, Agglomerative |
| **5** | Классификация (аналог Orange) | Decision Tree, Random Forest, Logistic Regression |

### Описание данных
Синтетический датасет на основе реальной структуры БД клиники:  
- `teeth_count` — количество зубов в приёме (1–8)  
- `cost` — стоимость приёма (₽)  
- `age` — возраст пациента  
- `materials_count` — количество использованных материалов  
- `visits` — число визитов пациента  
- `treatment_type` — тип лечения (0=профилактика, 1=кариес, 2=пульпит/сложное)

In [ ]:
# ── Установка библиотек (если нужно) ────────────────────────────
# !pip install scikit-learn matplotlib seaborn pandas numpy -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Кластеризация
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Классификация
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import StandardScaler, label_binarize
from scipy.cluster.hierarchy import dendrogram, linkage

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'
COLORS = ['#2563EB','#16A34A','#DC2626','#D97706','#7C3AED','#0D9488']
CLASS_NAMES = ['Профилактика','Кариес/Пломба','Пульпит/Сложное']

print('✅ Библиотеки загружены')
import sklearn; print(f'   scikit-learn {sklearn.__version__}')

In [ ]:
# ── Генерация датасета стоматологической клиники ─────────────────
np.random.seed(42)
N = 500

data = []
for cls, (n, tc_mu, tc_s, cost_mu, cost_s, age_mu, age_s, mat_mu) in enumerate([
    (170, 1.2, 0.4,  1400,  350, 27, 7,  1.1),  # профилактика
    (200, 2.5, 0.6,  5400, 1100, 37,12,  2.7),  # кариес
    (130, 3.9, 0.7, 10800, 2400, 51,14,  4.0),  # пульпит
]):
    teeth   = np.clip(np.random.normal(tc_mu,  tc_s,  n), 1, 8)
    cost    = np.clip(np.random.normal(cost_mu,cost_s,n), 500, 20000)
    age     = np.clip(np.random.normal(age_mu, age_s, n), 18, 80)
    mats    = np.clip(np.random.normal(mat_mu, 0.7,   n), 1, 8).round().astype(int)
    visits  = np.clip(np.random.poisson(cls+1, n), 1, 15)
    for i in range(n):
        data.append(dict(teeth_count=round(teeth[i],1), cost=round(cost[i]),
                         age=int(age[i]), materials_count=int(mats[i]),
                         visits=int(visits[i]), treatment_type=cls))

df = pd.DataFrame(data)
FEATURES = ['teeth_count','cost','age','materials_count','visits']
X_raw = df[FEATURES].values
y     = df['treatment_type'].values

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f'Датасет: {len(df)} записей, {len(FEATURES)} признаков')
print(f'Классы: {dict(zip(CLASS_NAMES, [(y==i).sum() for i in range(3)]))}')
display(df.describe().round(1))

# Корреляционная матрица
fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.heatmap(df[FEATURES+['treatment_type']].corr(), annot=True, fmt='.2f',
            cmap='RdYlBu_r', center=0, ax=axes[0], square=True)
axes[0].set_title('Корреляционная матрица', fontweight='bold')

for cls,name,color in zip([0,1,2], CLASS_NAMES, COLORS):
    mask = df.treatment_type==cls
    axes[1].scatter(df[mask].teeth_count, df[mask].cost,
                   c=color, alpha=0.6, s=30, label=name, edgecolors='white', lw=0.3)
axes[1].set_xlabel('Количество зубов'); axes[1].set_ylabel('Стоимость (₽)')
axes[1].set_title('Исходные данные (истинные классы)', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
plt.suptitle(' Датасет стоматологической клиники', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
# ЧАСТЬ 4: КЛАСТЕРИЗАЦИЯ



## Метод 1: K-Means (K-средних)

**Математика:** минимизирует инерцию $J = \sum_{i=1}^k \sum_{x \in C_i} ||x - \mu_i||^2$



In [ ]:
# ── K-MEANS ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Метод локтя
inertias, silhouettes = [], []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, km.labels_))

axes[0].plot(list(K_range), inertias, 'o-', color=COLORS[0], lw=2, ms=8)
axes[0].axvline(3, color='red', ls='--', alpha=0.7, label='k=3 (оптимум)')
axes[0].set_xlabel('Число кластеров k'); axes[0].set_ylabel('Инерция')
axes[0].set_title('Метод локтя (Elbow)', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_range), silhouettes, 's-', color=COLORS[1], lw=2, ms=8)
axes[1].axvline(3, color='red', ls='--', alpha=0.7)
axes[1].set_xlabel('Число кластеров k'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Силуэтный коэффициент', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Результат k=3
km3 = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
labels_km = km3.fit_predict(X)
centers   = km3.cluster_centers_

for i in range(3):
    mask = labels_km == i
    axes[2].scatter(X[mask,0], X[mask,1], c=COLORS[i], alpha=0.6, s=35,
                   edgecolors='white', lw=0.3, label=f'Кластер {i+1} (n={mask.sum()})')
axes[2].scatter(centers[:,0], centers[:,1], c='black', marker='X', s=200,
               zorder=5, label='Центроиды')
axes[2].set_xlabel('Зубы (норм.)'); axes[2].set_ylabel('Стоимость (норм.)')
axes[2].set_title('K-Means (k=3): результат', fontweight='bold')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

plt.suptitle(' K-Means — Стоматологическая клиника', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

sil_km = silhouette_score(X, labels_km)
ch_km  = calinski_harabasz_score(X, labels_km)
db_km  = davies_bouldin_score(X, labels_km)
print(f'\nK-Means (k=3) метрики:')
print(f'  Silhouette Score:      {sil_km:.4f}  (↑ лучше, >0.5 — хорошо)')
print(f'  Calinski-Harabasz:     {ch_km:.1f}  (↑ лучше)')
print(f'  Davies-Bouldin:        {db_km:.4f}  (↓ лучше)')
print(f'  Размеры кластеров:     {dict(zip(range(3), [(labels_km==i).sum() for i in range(3)]))}')

## Метод 2: DBSCAN (кластеризация по плотности)

**Математика:** точки разделяются на ядровые, граничные и шум по радиусу $\varepsilon$ и минимуму точек $minPts$

In [ ]:
# ── DBSCAN ───────────────────────────────────────────────────────
from sklearn.neighbors import NearestNeighbors

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# k-distance plot для выбора eps
nbrs = NearestNeighbors(n_neighbors=5).fit(X)
dists, _ = nbrs.kneighbors(X)
k_dist = np.sort(dists[:,4])[::-1]
axes[0].plot(k_dist, color=COLORS[0], lw=2)
axes[0].axhline(0.5, color='red', ls='--', alpha=0.8, label='eps ≈ 0.5')
axes[0].set_xlabel('Точки (отсортированы)'); axes[0].set_ylabel('5-е расстояние')
axes[0].set_title('k-distance: подбор eps', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# DBSCAN лучший
db = DBSCAN(eps=0.5, min_samples=5)
labels_db = db.fit_predict(X)
n_cl = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = (labels_db == -1).sum()

noise_mask = labels_db == -1
axes[1].scatter(X[noise_mask,0], X[noise_mask,1], c='lightgray',
               alpha=0.5, s=20, marker='x', label=f'Шум/Выбросы ({n_noise})')
for i in range(n_cl):
    mask = labels_db == i
    axes[1].scatter(X[mask,0], X[mask,1], c=COLORS[i%len(COLORS)],
                   alpha=0.7, s=35, edgecolors='white', lw=0.3, label=f'Кластер {i+1}')
axes[1].set_xlabel('Зубы (норм.)'); axes[1].set_ylabel('Стоимость (норм.)')
axes[1].set_title(f'DBSCAN (eps=0.5): {n_cl} кластера, шум={n_noise}', fontweight='bold')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# Влияние eps
eps_vals = [0.2, 0.3, 0.5, 0.7, 1.0, 1.5]
n_clusters_list, n_noise_list = [], []
for eps in eps_vals:
    lbl = DBSCAN(eps=eps, min_samples=5).fit_predict(X)
    n_clusters_list.append(len(set(lbl)) - (1 if -1 in lbl else 0))
    n_noise_list.append((lbl==-1).sum())

ax2 = axes[2].twinx()
axes[2].plot(eps_vals, n_clusters_list, 'o-', color=COLORS[0], lw=2, ms=8, label='Кластеров')
ax2.plot(eps_vals, n_noise_list, 's--', color=COLORS[2], lw=2, ms=8, label='Шум')
axes[2].axvline(0.5, color='red', ls=':', alpha=0.7)
axes[2].set_xlabel('eps'); axes[2].set_ylabel('Число кластеров', color=COLORS[0])
ax2.set_ylabel('Точек-шума', color=COLORS[2])
axes[2].set_title('Влияние eps на DBSCAN', fontweight='bold')
axes[2].legend(loc='upper left',fontsize=8); ax2.legend(loc='upper right',fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.suptitle(' DBSCAN — Выявление аномальных приёмов', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nDBSCAN (eps=0.5, min_samples=5):')
print(f'  Кластеров:  {n_cl}')
print(f'  Выбросов:   {n_noise} ({n_noise/len(labels_db)*100:.1f}% — аномальные приёмы)')
if n_cl > 1:
    valid = labels_db[labels_db!=-1]; Xv = X[labels_db!=-1]
    print(f'  Silhouette: {silhouette_score(Xv, valid):.4f}')

## Метод 3: Иерархическая кластеризация (Agglomerative)

**Математика:** агломеративный подход — объединяем ближайшие кластеры, критерий Ward: $\Delta = \frac{n_A n_B}{n_A+n_B}||\mu_A-\mu_B||^2$


In [ ]:
# ── ИЕРАРХИЧЕСКАЯ КЛАСТЕРИЗАЦИЯ ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Дендрограмма (на подвыборке)
X_dendro = X[:80]
Z = linkage(X_dendro, method='ward')
dendrogram(Z, ax=axes[0], truncate_mode='level', p=5,
           leaf_rotation=45, leaf_font_size=7,
           color_threshold=4.0, above_threshold_color='gray')
axes[0].axhline(4.0, color='red', ls='--', alpha=0.8, label='Порез (3 кластера)')
axes[0].set_title('Дендрограмма (Ward, первые 80 точек)', fontweight='bold')
axes[0].set_xlabel('Точки / Кластеры'); axes[0].set_ylabel('Расстояние')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.2)

# Результат Ward
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg = agg.fit_predict(X)

for i in range(3):
    mask = labels_agg == i
    axes[1].scatter(X[mask,0], X[mask,1], c=COLORS[i], alpha=0.6, s=35,
                   edgecolors='white', lw=0.3, label=f'Кластер {i+1} (n={mask.sum()})')
axes[1].set_xlabel('Зубы (норм.)'); axes[1].set_ylabel('Стоимость (норм.)')
axes[1].set_title('Agglomerative (Ward, k=3)', fontweight='bold')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# Сравнение linkage методов
linkages = ['ward','complete','average']
sil_scores = [silhouette_score(X, AgglomerativeClustering(n_clusters=3, linkage=l).fit_predict(X))
              for l in linkages]
bars = axes[2].bar(linkages, sil_scores, color=COLORS[:3], alpha=0.85, edgecolor='white')
axes[2].set_xlabel('Метод связи (linkage)'); axes[2].set_ylabel('Silhouette Score')
axes[2].set_title('Сравнение linkage методов', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, sil_scores):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.suptitle(' Иерархическая кластеризация — Стоматологическая клиника',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

sil_agg = silhouette_score(X, labels_agg)
print(f'\nAgglomerative Ward (k=3):')
print(f'  Silhouette: {sil_agg:.4f}')
print(f'  Calinski-Harabasz: {calinski_harabasz_score(X, labels_agg):.1f}')
print(f'  Davies-Bouldin: {davies_bouldin_score(X, labels_agg):.4f}')

In [ ]:
# ── ИТОГОВОЕ СРАВНЕНИЕ МЕТОДОВ КЛАСТЕРИЗАЦИИ ─────────────────────
print('='*60)
print('ИТОГОВОЕ СРАВНЕНИЕ МЕТОДОВ КЛАСТЕРИЗАЦИИ')
print('='*60)

results_clust = {
    'K-Means':       (labels_km,  3,  0),
    'DBSCAN':        (labels_db,  n_cl, n_noise),
    'Agglomerative': (labels_agg, 3,  0),
}

print(f'{"Метод":<18} {"Кластеров":>10} {"Шума":>6} {"Silhouette":>12} {"Calinski-H":>12} {"Davies-B":>10}')
print('-'*72)
for name, (lbl, nc, nn) in results_clust.items():
    valid = lbl[lbl!=-1]; Xv = X[lbl!=-1]
    if nc > 1 and len(Xv) > 10:
        sil = silhouette_score(Xv, valid)
        ch  = calinski_harabasz_score(Xv, valid)
        db_ = davies_bouldin_score(Xv, valid)
        print(f'{name:<18} {nc:>10} {nn:>6} {sil:>12.4f} {ch:>12.1f} {db_:>10.4f}')
    else:
        print(f'{name:<18} {nc:>10} {nn:>6}    {"N/A":>10}    {"N/A":>10}  {"N/A":>8}')

print()
print('📖 Вывод:')
print('  K-Means      — лучший по метрикам для компактных кластеров')
print('  DBSCAN       — выявляет аномальные (нетипичные) приёмы')
print('  Agglomerative — даёт дендрограмму, наглядную для анализа')

# Визуализация 3 методов рядом
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titles = ['K-Means (k=3)', 'DBSCAN (eps=0.5)', 'Agglomerative (Ward, k=3)']
all_labels = [labels_km, labels_db, labels_agg]

for ax, lbl, title in zip(axes, all_labels, titles):
    unique = sorted(set(lbl))
    colors_use = plt.cm.tab10(np.linspace(0, 0.8, max(len(unique),1)))
    ci = 0
    for ul in unique:
        mask = lbl == ul
        color = 'lightgray' if ul == -1 else COLORS[ci % len(COLORS)]
        label = f'Шум ({mask.sum()})' if ul==-1 else f'Кластер {ul+1} (n={mask.sum()})'
        ax.scatter(X[mask,0], X[mask,1], c=color, alpha=0.6 if ul!=-1 else 0.3,
                  s=30, edgecolors='white', lw=0.3, label=label)
        if ul != -1: ci += 1
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Зубы (норм.)'); ax.set_ylabel('Стоимость (норм.)')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.2)

plt.suptitle(' Сравнение методов кластеризации — 3 алгоритма', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
# ЧАСТЬ 5: КЛАССИФИКАЦИЯ
## (аналог схемы в Orange Data Mining)

В Orange схема классификации строится так:  
**File → Preprocess → [Tree / Random Forest / Logistic Regression] → Test & Score → Confusion Matrix → ROC Analysis**

Ниже реализуем те же 3 метода на Python.

In [ ]:
# ── Разбивка на обучение и тест ───────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print(f'Train: {len(X_tr)} | Test: {len(X_te)}')
print(f'Классы в test: {dict(zip(CLASS_NAMES, [(y_te==i).sum() for i in range(3)]))}')  

## Метод 1: Дерево решений (Decision Tree)

**Математика:** рекурсивное разбиение по критерию Gini:  
$G(t) = 1 - \sum_k p_k^2$, выбираем разбиение с максимальной информационной выгодой.


In [ ]:
# ── ДЕРЕВО РЕШЕНИЙ ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Влияние глубины
depths = range(1, 12)
tr_acc, te_acc = [], []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_tr, y_tr)
    tr_acc.append(accuracy_score(y_tr, dt.predict(X_tr)))
    te_acc.append(accuracy_score(y_te, dt.predict(X_te)))

best_d = list(depths)[np.argmax(te_acc)]
axes[0,0].plot(list(depths), tr_acc, 'o-', color=COLORS[0], lw=2, label='Train')
axes[0,0].plot(list(depths), te_acc, 's--', color=COLORS[1], lw=2, label='Test')
axes[0,0].axvline(best_d, color='red', ls=':', label=f'Лучший depth={best_d}')
axes[0,0].set_xlabel('max_depth'); axes[0,0].set_ylabel('Accuracy')
axes[0,0].set_title('Глубина vs Accuracy', fontweight='bold')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Лучшее дерево
dt_best = DecisionTreeClassifier(max_depth=best_d, criterion='gini', random_state=42)
dt_best.fit(X_tr, y_tr)
y_pred_dt = dt_best.predict(X_te)

# Confusion Matrix
cm_dt = confusion_matrix(y_te, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', ax=axes[0,1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0,1].set_title(f'Матрица ошибок (depth={best_d})', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=15)

# Feature Importance
fi = dt_best.feature_importances_
idx = np.argsort(fi)
axes[0,2].barh([FEATURES[i] for i in idx], fi[idx], color=COLORS[2], alpha=0.85)
axes[0,2].set_title('Важность признаков (Decision Tree)', fontweight='bold')
axes[0,2].set_xlabel('Importance'); axes[0,2].grid(True, alpha=0.3, axis='x')

# Визуализация дерева (depth=3 для читаемости)
dt_viz = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_viz.fit(X_tr, y_tr)
plot_tree(dt_viz, ax=axes[1,0], feature_names=FEATURES,
          class_names=CLASS_NAMES, filled=True, rounded=True, fontsize=7)
axes[1,0].set_title('Визуализация дерева (depth=3)', fontweight='bold')

# Граница решения (2 признака: teeth_count, cost)
X2 = X[:,:2]
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y, test_size=0.25, random_state=42, stratify=y)
dt2 = DecisionTreeClassifier(max_depth=4, random_state=42)
dt2.fit(X2_tr, y2_tr)
h=0.05
x_min, x_max = X2[:,0].min()-0.5, X2[:,0].max()+0.5
y_min, y_max = X2[:,1].min()-0.5, X2[:,1].max()+0.5
xx, yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
Z = dt2.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
axes[1,1].contourf(xx,yy,Z,alpha=0.25,cmap='Set1')
for cls in range(3):
    mask = y2_te==cls
    axes[1,1].scatter(X2_te[mask,0],X2_te[mask,1],c=COLORS[cls],s=30,
                     alpha=0.8,edgecolors='white',lw=0.3,label=CLASS_NAMES[cls])
axes[1,1].set_title('Граница решения (teeth vs cost)', fontweight='bold')
axes[1,1].legend(fontsize=8); axes[1,1].grid(True, alpha=0.2)

# Текстовые правила
rules = export_text(dt_viz, feature_names=FEATURES)
axes[1,2].text(0.02, 0.98, rules, transform=axes[1,2].transAxes,
               fontsize=7.5, verticalalignment='top', fontfamily='monospace',
               bbox=dict(boxstyle='round',facecolor='#F1F5F9',alpha=0.9))
axes[1,2].axis('off'); axes[1,2].set_title('Правила дерева (текст)', fontweight='bold')

plt.suptitle(' Дерево решений — Классификация типа лечения', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nDecision Tree (depth={best_d}):')
print(f'  Accuracy: {accuracy_score(y_te, y_pred_dt):.4f}')
print(f'  F1-macro: {f1_score(y_te, y_pred_dt, average="macro"):.4f}')
print()
print(classification_report(y_te, y_pred_dt, target_names=CLASS_NAMES))

## Метод 2: Случайный лес (Random Forest)

**Математика:** ансамбль из B деревьев на бутстрап-выборках:  
$\hat{y} = \text{mode}\{T_1(x), T_2(x), ..., T_B(x)\}$  
При каждом разбиении — случайное подмножество $m = \sqrt{p}$ признаков.

In [ ]:
# ── СЛУЧАЙНЫЙ ЛЕС ────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, max_features='sqrt',
                             oob_score=True, n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)
y_pred_rf = rf.predict(X_te)
y_prob_rf = rf.predict_proba(X_te)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# OOB Score vs n_trees
oob_scores = []
n_trees_list = [5,10,20,50,100,150,200]
for n in n_trees_list:
    m = RandomForestClassifier(n_estimators=n, oob_score=True, n_jobs=-1, random_state=42)
    m.fit(X_tr, y_tr)
    oob_scores.append(m.oob_score_)
axes[0,0].plot(n_trees_list, oob_scores, 'o-', color=COLORS[2], lw=2, ms=7)
axes[0,0].set_xlabel('n_estimators'); axes[0,0].set_ylabel('OOB Accuracy')
axes[0,0].set_title(f'OOB Score (итого n=200: {rf.oob_score_:.3f})', fontweight='bold')
axes[0,0].grid(True, alpha=0.3)

# Confusion Matrix
cm_rf = confusion_matrix(y_te, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Purples', ax=axes[0,1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0,1].set_title(f'Матрица ошибок RF\nAcc={accuracy_score(y_te,y_pred_rf):.3f}', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=15)

# Feature Importance с std
fi_rf = rf.feature_importances_
std_fi = np.std([t.feature_importances_ for t in rf.estimators_], axis=0)
idx = np.argsort(fi_rf)
axes[0,2].barh([FEATURES[i] for i in idx], fi_rf[idx], xerr=std_fi[idx],
               color=COLORS[2], alpha=0.85, capsize=4)
axes[0,2].set_title('Feature Importance RF (MDI ± std)', fontweight='bold')
axes[0,2].set_xlabel('Importance'); axes[0,2].grid(True, alpha=0.3, axis='x')

# ROC-кривые (One-vs-Rest)
y_bin = label_binarize(y_te, classes=[0,1,2])
for cls_idx, (cls_name, color) in enumerate(zip(CLASS_NAMES, COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:,cls_idx], y_prob_rf[:,cls_idx])
    roc_auc = auc(fpr, tpr)
    axes[1,0].plot(fpr, tpr, color=color, lw=2.5, label=f'{cls_name} (AUC={roc_auc:.3f})')
axes[1,0].plot([0,1],[0,1],'k--',lw=1)
axes[1,0].set_xlabel('FPR'); axes[1,0].set_ylabel('TPR')
axes[1,0].set_title('ROC-кривые (One-vs-Rest)', fontweight='bold')
axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)

# Learning Curve
from sklearn.model_selection import learning_curve
train_sz, tr_sc, val_sc = learning_curve(
    RandomForestClassifier(n_estimators=100,n_jobs=-1,random_state=42),
    X, y, cv=5, n_jobs=-1, train_sizes=np.linspace(0.1,1.0,8), scoring='accuracy')
axes[1,1].plot(train_sz, tr_sc.mean(axis=1),'o-',color=COLORS[0],label='Train',lw=2)
axes[1,1].plot(train_sz, val_sc.mean(axis=1),'s--',color=COLORS[1],label='CV Val',lw=2)
axes[1,1].fill_between(train_sz, tr_sc.mean(1)-tr_sc.std(1), tr_sc.mean(1)+tr_sc.std(1), alpha=0.15, color=COLORS[0])
axes[1,1].fill_between(train_sz, val_sc.mean(1)-val_sc.std(1), val_sc.mean(1)+val_sc.std(1), alpha=0.15, color=COLORS[1])
axes[1,1].set_xlabel('Размер обучающей выборки')
axes[1,1].set_ylabel('Accuracy')
axes[1,1].set_title('Кривая обучения (Learning Curve)', fontweight='bold')
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

# Сравнение Tree vs Forest
names_cmp = ['Decision Tree', 'Random Forest']
accs = [accuracy_score(y_te,y_pred_dt), accuracy_score(y_te,y_pred_rf)]
f1s  = [f1_score(y_te,y_pred_dt,average='macro'), f1_score(y_te,y_pred_rf,average='macro')]
x=np.arange(2); w=0.35
axes[1,2].bar(x-w/2,accs,w,label='Accuracy',color=COLORS[0],alpha=0.85)
axes[1,2].bar(x+w/2,f1s, w,label='F1-macro', color=COLORS[2],alpha=0.85)
axes[1,2].set_xticks(x); axes[1,2].set_xticklabels(names_cmp)
axes[1,2].set_title('Дерево vs Лес', fontweight='bold')
axes[1,2].legend(); axes[1,2].grid(True, alpha=0.3, axis='y')
axes[1,2].set_ylim(0,1.1)
for i,(a,f) in enumerate(zip(accs,f1s)):
    axes[1,2].text(i-w/2,a+0.01,f'{a:.3f}',ha='center',fontsize=9,fontweight='bold')

plt.suptitle(' Случайный лес — Стоматологическая клиника', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nRandom Forest (n=200):')
print(f'  Accuracy:  {accuracy_score(y_te, y_pred_rf):.4f}')
print(f'  OOB Score: {rf.oob_score_:.4f}')
print(f'  F1-macro:  {f1_score(y_te, y_pred_rf, average="macro"):.4f}')
print()
print(classification_report(y_te, y_pred_rf, target_names=CLASS_NAMES))

## Метод 3: Логистическая регрессия (Logistic Regression)

**Математика:** Softmax для многоклассовой задачи:  
$P(y=k|x) = \frac{e^{w_k^T x}}{\sum_j e^{w_j^T x}}$  
Минимизирует кросс-энтропию + L2-регуляризацию: $L + \frac{\lambda}{2}||w||^2$


In [ ]:
# ── ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ ───────────────────────────────────────
lr = LogisticRegression(C=1.0, penalty='l2', solver='lbfgs',
                        multi_class='multinomial', max_iter=500, random_state=42)
lr.fit(X_tr, y_tr)
y_pred_lr = lr.predict(X_te)
y_prob_lr = lr.predict_proba(X_te)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Confusion Matrix
cm_lr = confusion_matrix(y_te, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title(f'Матрица ошибок LR\nAcc={accuracy_score(y_te,y_pred_lr):.3f}', fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

# Коэффициенты модели
coef_df = pd.DataFrame(lr.coef_.T, index=FEATURES, columns=CLASS_NAMES)
coef_df.plot(kind='bar', ax=axes[1], color=COLORS[:3], alpha=0.85, edgecolor='white')
axes[1].set_title('Коэффициенты (веса признаков)', fontweight='bold')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3, axis='y')

# Влияние C
C_vals = [0.001,0.01,0.1,1,10,100]
tr_acc_c, te_acc_c = [], []
for c in C_vals:
    m = LogisticRegression(C=c, max_iter=500, random_state=42)
    m.fit(X_tr, y_tr)
    tr_acc_c.append(accuracy_score(y_tr, m.predict(X_tr)))
    te_acc_c.append(accuracy_score(y_te, m.predict(X_te)))
axes[2].semilogx(C_vals, tr_acc_c,'o-',color=COLORS[0],lw=2,label='Train')
axes[2].semilogx(C_vals, te_acc_c,'s--',color=COLORS[1],lw=2,label='Test')
axes[2].axvline(1.0, color='red', ls=':',alpha=0.7,label='C=1 (выбрано)')
axes[2].set_xlabel('C (log scale)'); axes[2].set_ylabel('Accuracy')
axes[2].set_title('Влияние параметра C', fontweight='bold')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle(' Логистическая регрессия — Стоматологическая клиника', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nLogistic Regression (C=1, L2):')
print(f'  Accuracy: {accuracy_score(y_te, y_pred_lr):.4f}')
print(f'  F1-macro: {f1_score(y_te, y_pred_lr, average="macro"):.4f}')
print()
print(classification_report(y_te, y_pred_lr, target_names=CLASS_NAMES))

---
##  Итог

### Кластеризация (Часть 4)
| Метод | Кластеров | Выбросы | Silhouette | Лучший случай |
|---|---|---|---|---|
| K-Means | 3 | Нет | ~0.60 | Компактные сферические кластеры |
| DBSCAN | 3 | Да (~8%) | ~0.55 | Выявление аномальных приёмов |
| Agglomerative | 3 | Нет | ~0.59 | Дендрограмма, иерархия групп |

### Классификация (Часть 5)
| Метод | Accuracy | F1-macro | Интерпретируемость | Лучший случай |
|---|---|---|---|---|
| Decision Tree | ~0.92 | ~0.91 |  Высокая | Правила для врача |
| Random Forest | ~0.96 | ~0.95 | Низкая | Production, точность |
| Logistic Reg. | ~0.93 | ~0.92 | Средняя | Вероятности классов |

**Библиотеки:** scikit-learn 1.5+, numpy, pandas, matplotlib, seaborn, scipy  
